# Healthcare data types: records, images, signals, notes and omics: worked solution

Worked solution for the exercise in the lesson **[Healthcare data types: records, images, signals, notes and omics](https://ramkumarmd.com/medical-ai/foundations/healthcare-data-types)** (*Foundations* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](02-healthcare-data-types.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** Healthcare data comes in five broad types: structured records (diagnoses, medicines, lab results), medical images, physiological signals such as ECGs, free-text clinical notes, and omics data such as genomes. Each needs different methods, and all of them reflect how care was delivered and documented, not just the patient's biology. Knowing how a dataset was produced is the first step in any AI project.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Structured EHR data: an event log, not a spreadsheet

Synthetic lab results for five made-up patients, one row per measurement.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

rows = []
for patient_id in range(1, 6):
    n_tests = rng.integers(2, 6)
    days = np.sort(rng.choice(365, size=n_tests, replace=False))
    for day in days:
        test = rng.choice(["hba1c", "glucose", "creatinine"], p=[0.4, 0.4, 0.2])
        value = {"hba1c": rng.normal(7.5, 1.2), "glucose": rng.normal(150, 35),
                 "creatinine": rng.normal(1.0, 0.25)}[test]
        rows.append({"patient_id": patient_id, "day": int(day),
                     "test": test, "value": round(value, 1)})

labs = pd.DataFrame(rows)
print(labs.head(8))
print(f"\n{len(labs)} measurements for {labs.patient_id.nunique()} patients")

#    patient_id  day        test  value
# 0           1  210  creatinine    0.9
# 1           1  225     glucose  162.5
# 2           1  247     glucose  174.3
# 3           1  283     glucose  104.9
# 4           1  325       hba1c    6.0
# 5           2  187  creatinine    0.6
# 6           2  354       hba1c    7.7
# 7           3    1       hba1c    7.6
#
# 20 measurements for 5 patients

### From events to one row per patient

Continuing from the previous block: the latest value of each test per patient.

In [ ]:
latest = (labs.sort_values("day")
              .groupby(["patient_id", "test"])["value"].last()
              .unstack("test"))
print(latest)
print("\nMissing values per test:")
print(latest.isna().sum())

# test        creatinine  glucose  hba1c
# patient_id
# 1                  0.9    104.9    6.0
# 2                  0.6      NaN    7.7
# 3                  1.4    220.0    7.6
# 4                  NaN      NaN    5.1
# 5                  1.0    186.8    6.0
#
# Missing values per test:
# creatinine    1
# glucose       2
# hba1c         0

## Exercise

**Goal:** Generate synthetic patient records with Synthea (or use its sample data), load them with pandas, and build a one-row-per-patient table with explicit choices about time windows and missing values.

**Data:** [Synthea synthetic patient records (CSV export)](https://github.com/synthetichealth/synthea) (Synthea is Apache 2.0; generated records are synthetic)

### Step 1

Load the Synthea patients.csv and observations.csv files (the notebook downloads a small sample).

In [ ]:
# Synthea sample data (synthetic patients, Apache 2.0), pinned to the April 2020 release so the numbers below stay stable.
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd

URL = "https://synthetichealth.github.io/synthea-sample-data/downloads/synthea_sample_data_csv_apr2020.zip"
zip_path = Path("data/synthea_csv_apr2020.zip")
zip_path.parent.mkdir(exist_ok=True)
if not zip_path.exists():
    urlretrieve(URL, zip_path)  # about 9 MB

with zipfile.ZipFile(zip_path) as z:
    patients = pd.read_csv(z.open("csv/patients.csv"), usecols=["Id", "BIRTHDATE", "DEATHDATE", "GENDER"],
                           parse_dates=["BIRTHDATE", "DEATHDATE"])
    observations = pd.read_csv(z.open("csv/observations.csv"), parse_dates=["DATE"])

print(f"{len(patients):,} patients, {len(observations):,} observations")
print(observations.head(3))

### Step 2

Filter observations to HbA1c, glucose and creatinine using their LOINC codes.

In [ ]:
# Synthea codes glucose as LOINC 2339-0 (Glucose [Mass/volume] in Blood), not 2345-7 (in Serum or Plasma).
# Checking which codes a source actually uses, before filtering, is part of the job.
LABS = {"4548-4": "hba1c", "2339-0": "glucose", "2345-7": "glucose", "38483-4": "creatinine"}
labs = observations[observations.CODE.isin(LABS)].copy()
labs["test"] = labs.CODE.map(LABS)
labs["value"] = pd.to_numeric(labs.VALUE, errors="coerce")
labs["DATE"] = labs.DATE.dt.tz_localize(None)

print(labs.groupby(["test", "CODE", "UNITS"]).size())

### Step 3

For each patient, compute the latest value and its age in days at a chosen index date.

In [ ]:
# Index date: 1 January 2020. Only results on or before it count, otherwise the table "knows the future".
# Patients who died before the index date are excluded: they could not have a prediction made for them.
INDEX_DATE = pd.Timestamp("2020-01-01")
alive = patients[(patients.BIRTHDATE < INDEX_DATE) &
                 (patients.DEATHDATE.isna() | (patients.DEATHDATE > INDEX_DATE))]

before = labs[labs.DATE <= INDEX_DATE].sort_values("DATE")
last = before.groupby(["PATIENT", "test"]).last()
wide = last["value"].unstack("test")
days_ago = (INDEX_DATE - last["DATE"]).dt.days.unstack("test").add_suffix("_days_ago")

table = (alive.set_index("Id")[["BIRTHDATE", "GENDER"]]
         .join(wide).join(days_ago))
table["age"] = ((INDEX_DATE - table.BIRTHDATE).dt.days / 365.25).astype(int)
table = table.drop(columns="BIRTHDATE")
print(f"{len(table):,} patients alive at the index date")
print(table.dropna(subset=["hba1c"]).head().round(1))   # a few patients who have an HbA1c

The `_days_ago` columns matter as much as the values: an HbA1c from last month and one from
eight years ago are not the same piece of information, but a model sees only a number unless you tell it.

### Step 4

Count how many patients have each test missing and compare missingness by age group.

In [ ]:
tests = ["hba1c", "glucose", "creatinine"]
print("Missing per test:")
print(table[tests].isna().sum())

table["age_group"] = pd.cut(table.age, bins=[0, 18, 40, 65, 120], right=False,
                            labels=["<18", "18-39", "40-64", "65+"])
missing_by_age = table.groupby("age_group", observed=True)[tests].agg(lambda s: s.isna().mean())
print("\nShare missing by age group:")
print(missing_by_age.round(2))

Missingness is far from random: in this sample no child has any of the three tests, while adults
aged 65 and over are the most likely to have them. Whether a test was ordered is itself a signal about the patient (and about the clinician),
so "missing" should be modelled explicitly, not silently filled with an average.

### Step 5

Write down three ways the resulting table could mislead a model.

Three ways this table could mislead a model (example answers):

1. **Informative missingness.** Tests are ordered when a clinician is worried. A model can learn that
   "has an HbA1c" predicts diabetes without learning anything about glucose metabolism, and it will
   fail in a setting with universal screening.
2. **Stale values.** The latest value may be years old. Without the `_days_ago` columns, a 2012 HbA1c
   looks as current as one from last week.
3. **Synthetic and single-region data.** Synthea simulates Massachusetts residents from published
   disease models. Its distributions, testing patterns and units are not those of any real hospital,
   so anything learned here must be revalidated on real (properly governed) data before it means anything.

Other good answers: one-row-per-patient hides trends (a falling HbA1c and a rising one can have the same
latest value); the index-date choice changes who is included; codes differ between sources (2339-0 vs 2345-7).

---

Lesson: [https://ramkumarmd.com/medical-ai/foundations/healthcare-data-types](https://ramkumarmd.com/medical-ai/foundations/healthcare-data-types). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.